# F1 Analytics


## 1. Dataset Selection

### Dataset Name
**Formula One (F1) Historical Race and Session Results Dataset, 2019–2024**

### Source
The dataset was obtained from the **Jolpica-F1 database**, an open-source project providing historical Formula One data. The data was downloaded as CSV files from the project's database dumps and is being analysed locally using Python and Pandas.

Source: https://github.com/jolpica/jolpica-f1

### Why I Chose It
I chose this dataset because of my interest in Formula One, particularly the engineering and aerodynamic developments that influence car performance. The sport provides an opportunity to explore how changes in technical regulations may be associated with differences in race outcomes.

The dataset contains information about drivers, teams, race results, starting grid positions, points and individual sessions. This makes it suitable for data cleaning, manipulation, statistical analysis and visualisation using Python.

I selected the period from 2019 to 2024 to compare three seasons before the major 2022 aerodynamic regulation changes with the three seasons that followed. This provides a consistent time window for investigating patterns across the two periods.

### Potential Question to Explore
**Did the relationship between qualifying position and race finishing position change after the 2022 Formula One regulations?**

To investigate this question, I will compare qualifying and finishing positions across the 2019–2021 and 2022–2024 periods. I will use statistical analysis and visualisations to examine whether qualifying performance became more or less strongly associated with race results following the regulatory changes.

The analysis will identify patterns in the data rather than assume that regulation changes alone caused any differences observed.



## 2. Initital Data Exploration

In [17]:
import pandas as pd
from fastf1.ergast import Ergast
from pathlib import Path


In [18]:
# check the files are loaded correctly
data_folder = Path("f1-csv")
csv_files = list(data_folder.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print(file.name)

Number of CSV files: 18
formula_one_baseteam.csv
formula_one_championshipadjustment.csv
formula_one_championshipsystem.csv
formula_one_circuit.csv
formula_one_driver.csv
formula_one_driverchampionship.csv
formula_one_lap.csv
formula_one_penalty.csv
formula_one_pitstop.csv
formula_one_pointsystem.csv
formula_one_round.csv
formula_one_roundentry.csv
formula_one_season.csv
formula_one_session.csv
formula_one_sessionentry.csv
formula_one_team.csv
formula_one_teamchampionship.csv
formula_one_teamdriver.csv


In [19]:
# Load the tables I need into Pandas
baseteam = pd.read_csv(data_folder / "formula_one_baseteam.csv")
sessionentry = pd.read_csv(data_folder / "formula_one_sessionentry.csv")
session = pd.read_csv(data_folder / "formula_one_session.csv")
rounds = pd.read_csv(data_folder / "formula_one_round.csv")
seasons = pd.read_csv(data_folder / "formula_one_season.csv")
drivers = pd.read_csv(data_folder / "formula_one_driver.csv")
teams = pd.read_csv(data_folder / "formula_one_team.csv")

In [32]:
# Inspect first rows
print("SESSION ENTRY")
sessionentry.head()

print("SESSION")
session.head()

print("ROUND")
rounds.head()

print("SEASON")
seasons.head()

# Here, I noticed that the seasons start from the inaugural season in 1950

SESSION ENTRY
SESSION
ROUND
SEASON


,id,api_id,championship_system_id,wikipedia,year
0,1,season_oPqQEC4b,1,https://en.wikipedia.org/wiki/1950_Formula_One...,1950
1,2,season_nH07GTj5,1,https://en.wikipedia.org/wiki/1951_Formula_One...,1951
2,3,season_AkHDeRX4,1,https://en.wikipedia.org/wiki/1952_Formula_One...,1952
3,4,season_E63rvZZO,1,https://en.wikipedia.org/wiki/1953_Formula_One...,1953
4,5,season_B36NWfA1,2,https://en.wikipedia.org/wiki/1954_Formula_One...,1954


In [26]:
# Work out column names, data types and no. of rows and columns
for name, df in {
    "sessionentry": sessionentry,
    "session": session,
    "rounds": rounds,
    "seasons": seasons,
    "drivers": drivers,
    "teams": teams
}.items():
    print(f"\n--- {name} ---")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.dtypes)


--- sessionentry ---
Shape: (51685, 16)
Columns: ['id', 'api_id', 'detail', 'driver_of_the_day_percentage', 'driver_of_the_day_winner', 'fastest_lap_rank', 'grid', 'is_classified', 'is_eligible_for_points', 'laps_completed', 'points', 'position', 'round_entry_id', 'session_id', 'status', 'time']
id                                int64
api_id                           object
detail                           object
driver_of_the_day_percentage    float64
driver_of_the_day_winner        float64
fastest_lap_rank                float64
grid                            float64
is_classified                    object
is_eligible_for_points           object
laps_completed                  float64
points                          float64
position                        float64
round_entry_id                    int64
session_id                        int64
status                          float64
time                             object
dtype: object

--- session ---
Shape: (5130, 13)
Columns: ['id

In [22]:
for name, df in {
    "sessionentry": sessionentry,
    "session": session,
    "rounds": rounds,
    "seasons": seasons,
    "drivers": drivers,
    "teams": teams
}.items():
    print(f"\n{name}:")
    print(df.columns.tolist())
    


sessionentry:
['id', 'api_id', 'detail', 'driver_of_the_day_percentage', 'driver_of_the_day_winner', 'fastest_lap_rank', 'grid', 'is_classified', 'is_eligible_for_points', 'laps_completed', 'points', 'position', 'round_entry_id', 'session_id', 'status', 'time']

session:
['id', 'api_id', 'completed_distance', 'completed_laps', 'has_time_data', 'is_cancelled', 'number', 'point_system_id', 'round_id', 'scheduled_laps', 'timestamp', 'timezone', 'type']

rounds:
['id', 'api_id', 'circuit_id', 'date', 'is_cancelled', 'name', 'number', 'race_number', 'season_id', 'wikipedia']

seasons:
['id', 'api_id', 'championship_system_id', 'wikipedia', 'year']

drivers:
['id', 'abbreviation', 'api_id', 'country_code', 'date_of_birth', 'forename', 'nationality', 'permanent_car_number', 'reference', 'surname', 'wikipedia']

teams:
['id', 'api_id', 'base_team_id', 'country_code', 'name', 'nationality', 'primary_color', 'reference', 'wikipedia']


## 3. Data Cleaning

My data is split into multiple CSVs so inspection of the tables to work out primary and foreign key is needed

In [31]:
roundentry = pd.read_csv(data_folder / "formula_one_roundentry.csv")

print("Shape:", roundentry.shape)
print("Columns:", roundentry.columns.tolist())
roundentry.head()

Shape: (27663, 5)
Columns: ['id', 'api_id', 'car_number', 'round_id', 'team_driver_id']


,id,api_id,car_number,round_id,team_driver_id
0,1,roundentry_FUFWqiHA,2.0,1,1
1,2,roundentry_X2Oso05J,3.0,1,2
2,3,roundentry_wjjINPdb,4.0,1,3
3,4,roundentry_kg7fbLfc,14.0,1,4
4,5,roundentry_LAN7L55t,15.0,1,5


In [30]:
teamdriver = pd.read_csv(data_folder / "formula_one_teamdriver.csv")

print("Shape:", teamdriver.shape)
print("Columns:", teamdriver.columns.tolist())
teamdriver.head()

Shape: (3630, 6)
Columns: ['id', 'api_id', 'driver_id', 'role', 'season_id', 'team_id']


,id,api_id,driver_id,role,season_id,team_id
0,1,teamdriver_LSox06Od,1,NaN,1,1
1,2,teamdriver_bLowFOCJ,2,NaN,1,1
2,3,teamdriver_d8PF1G74,3,NaN,1,1
3,4,teamdriver_DuMzxcI2,4,NaN,1,5
4,5,teamdriver_0BTCGA9I,5,NaN,1,5


In [33]:
# Selection of the six seasons in scope
seasons_2019_2024 = seasons[
    seasons["year"].between(2019, 2024)
].copy()

# Select the rounds that match up to the years
rounds_2019_2024 = rounds[
    rounds["season_id"].isin(seasons_2019_2024["id"])
].copy()

# Select sessions belonging to those rounds
sessions_2019_2024 = session[
    session["round_id"].isin(rounds_2019_2024["id"])
].copy()

# Select the entries belonging to those sessions
entries_2019_2024 = sessionentry[
    sessionentry["session_id"].isin(sessions_2019_2024["id"])
].copy()

print("Seasons:", seasons_2019_2024["year"].min(),
      "to", seasons_2019_2024["year"].max())
print("Number of seasons:", seasons_2019_2024["year"].nunique())
print("Rounds:", len(rounds_2019_2024))
print("Sessions:", len(sessions_2019_2024))
print("Session entries:", len(entries_2019_2024))

Seasons: 2019 to 2024
Number of seasons: 6
Rounds: 128
Sessions: 920
Session entries: 9129


In [35]:
# Duplicate check:
print("Duplicate session entry IDs:",
      entries_2019_2024["id"].duplicated().sum())

print("Duplicate session IDs:",
      sessions_2019_2024["id"].duplicated().sum())

print("Duplicate round IDs:",
      rounds_2019_2024["id"].duplicated().sum())

print("Duplicate season IDs:",
      seasons_2019_2024["id"].duplicated().sum())

Duplicate session entry IDs: 0
Duplicate session IDs: 0
Duplicate round IDs: 0
Duplicate season IDs: 0


### Duplicate Checks

I checked the primary ID columns in the session entries, sessions, rounds and seasons tables for duplicate values. All four tables contained zero duplicate IDs.

This suggests that the IDs can be used to identify individual records in these tables. Further checks will be carried out after merging to ensure that each driver has the expected number of entries per session and that the joins have not created duplicate records.

In [40]:
# Inspection of relationship tables
roundentry.head()

,id,api_id,car_number,round_id,team_driver_id
0,1,roundentry_FUFWqiHA,2.0,1,1
1,2,roundentry_X2Oso05J,3.0,1,2
2,3,roundentry_wjjINPdb,4.0,1,3
3,4,roundentry_kg7fbLfc,14.0,1,4
4,5,roundentry_LAN7L55t,15.0,1,5


In [38]:
teamdriver.head()

,id,api_id,driver_id,role,season_id,team_id
0,1,teamdriver_LSox06Od,1,NaN,1,1
1,2,teamdriver_bLowFOCJ,2,NaN,1,1
2,3,teamdriver_d8PF1G74,3,NaN,1,1
3,4,teamdriver_DuMzxcI2,4,NaN,1,5
4,5,teamdriver_0BTCGA9I,5,NaN,1,5


In [41]:
roundentry.columns.tolist()

['id', 'api_id', 'car_number', 'round_id', 'team_driver_id']

In [42]:
teamdriver.columns.tolist()

['id', 'api_id', 'driver_id', 'role', 'season_id', 'team_id']

In [43]:
print("Driver columns:")
print(drivers.columns.tolist())

print("\nTeam columns:")
print(teams.columns.tolist())

print("\nSession types in 2019–2024:")
print(sessions_2019_2024["type"].value_counts(dropna=False))

Driver columns:
['id', 'abbreviation', 'api_id', 'country_code', 'date_of_birth', 'forename', 'nationality', 'permanent_car_number', 'reference', 'surname', 'wikipedia']

Team columns:
['id', 'api_id', 'base_team_id', 'country_code', 'name', 'nationality', 'primary_color', 'reference', 'wikipedia']

Session types in 2019–2024:
type
R      128
Q1     128
Q2     128
Q3     128
FP1    128
FP2    116
FP3    110
SR      18
SQ1     12
SQ2     12
SQ3     12
Name: count, dtype: int64


In [44]:
# Making a new DataFrame and merging all the tables

# Add season and race information to each session
session_info = sessions_2019_2024[
    ["id", "type", "round_id"]
].rename(columns={"id": "session_id"})

session_info = session_info.merge(
    rounds_2019_2024[["id", "name", "date", "season_id"]],
    left_on="round_id",
    right_on="id"
).drop(columns="id")

session_info = session_info.merge(
    seasons_2019_2024[["id", "year"]],
    left_on="season_id",
    right_on="id"
).drop(columns="id")


# Add session information to the results
f1_results = entries_2019_2024.merge(
    session_info,
    on="session_id",
    how="left"
)


# merge each result to its race entry
f1_results = f1_results.merge(
    roundentry[["id", "team_driver_id", "car_number"]].rename(
        columns={"id": "round_entry_id"}
    ),
    on="round_entry_id",
    how="left"
)


# merge each race entry to its driver and constructor IDs
f1_results = f1_results.merge(
    teamdriver[["id", "driver_id", "team_id"]].rename(
        columns={"id": "team_driver_id"}
    ),
    on="team_driver_id",
    how="left"
)


# Add drivers
f1_results = f1_results.merge(
    drivers[["id", "forename", "surname"]].rename(
        columns={"id": "driver_id"}
    ),
    on="driver_id",
    how="left"
)


# Add constructors
f1_results = f1_results.merge(
    teams[["id", "name"]].rename(
        columns={"id": "team_id", "name": "team_name"}
    ),
    on="team_id",
    how="left"
)

f1_results.head()

,id,api_id,detail,driver_of_the_day_percentage,driver_of_the_day_winner,fastest_lap_rank,grid,is_classified,is_eligible_for_points,laps_completed,...,date,season_id,year,team_driver_id,car_number,driver_id,team_id,forename,surname,team_name
0,38474,sessionentry_EeURiWah,Finished,NaN,NaN,1.0,2.0,t,t,58.0,...,2019-03-17,70,2019,3418,77.0,820,48,Valtteri,Bottas,Mercedes
1,38475,sessionentry_1jFQI4PW,NaN,NaN,NaN,NaN,NaN,NaN,t,NaN,...,2019-03-17,70,2019,3418,77.0,820,48,Valtteri,Bottas,Mercedes
2,38476,sessionentry_W61tGDAk,NaN,NaN,NaN,NaN,NaN,NaN,t,NaN,...,2019-03-17,70,2019,3418,77.0,820,48,Valtteri,Bottas,Mercedes
3,38477,sessionentry_PpDz0yOl,NaN,NaN,NaN,NaN,NaN,NaN,t,NaN,...,2019-03-17,70,2019,3418,77.0,820,48,Valtteri,Bottas,Mercedes
4,38478,sessionentry_xuvP3yxG,Finished,NaN,NaN,2.0,1.0,t,t,58.0,...,2019-03-17,70,2019,3419,44.0,795,48,Lewis,Hamilton,Mercedes


In [45]:
print("Rows in combined dataset:", len(f1_results))

print("\nMissing driver names:")
print(f1_results["forename"].isna().sum())

print("\nMissing constructor names:")
print(f1_results["team_name"].isna().sum())

print("\nSession types:")
print(f1_results["type"].value_counts())

Rows in combined dataset: 9129

Missing driver names:
0

Missing constructor names:
0

Session types:
type
R      2559
Q1     2556
Q2     1904
Q3     1255
SR      360
SQ1     220
SQ2     165
SQ3     110
Name: count, dtype: int64


In [46]:
# Check for how the dataset records qualifying positions

qualifying_check = f1_results[
    (f1_results["year"] == 2019) &
    (f1_results["type"].isin(["Q1", "Q2", "Q3"]))
]

qualifying_check[
    ["name", "type", "forename", "surname", "position", "grid"]
].head(20)

,name,type,forename,surname,position,grid
1,Australian Grand Prix,Q1,Valtteri,Bottas,2.0,NaN
2,Australian Grand Prix,Q2,Valtteri,Bottas,2.0,NaN
3,Australian Grand Prix,Q3,Valtteri,Bottas,2.0,NaN
5,Australian Grand Prix,Q1,Lewis,Hamilton,1.0,NaN
6,Australian Grand Prix,Q2,Lewis,Hamilton,1.0,NaN
7,Australian Grand Prix,Q3,Lewis,Hamilton,1.0,NaN
9,Australian Grand Prix,Q1,Max,Verstappen,4.0,NaN
10,Australian Grand Prix,Q2,Max,Verstappen,4.0,NaN
11,Australian Grand Prix,Q3,Max,Verstappen,4.0,NaN
13,Australian Grand Prix,Q1,Sebastian,Vettel,3.0,NaN


In [47]:
qualifying_check[
    qualifying_check["name"] == "Australian Grand Prix"
][["type", "forename", "surname", "position", "grid"]].sort_values(
    ["type", "position"]
)

,type,forename,surname,position,grid
5,Q1,Lewis,Hamilton,1.0,NaN
1,Q1,Valtteri,Bottas,2.0,NaN
13,Q1,Sebastian,Vettel,3.0,NaN
9,Q1,Max,Verstappen,4.0,NaN
17,Q1,Charles,Leclerc,5.0,NaN
57,Q1,Romain,Grosjean,6.0,NaN
21,Q1,Kevin,Magnussen,7.0,NaN
39,Q1,Lando,Norris,8.0,NaN
28,Q1,Kimi,Räikkönen,9.0,NaN
43,Q1,Sergio,Pérez,10.0,NaN


This means that the dataset only records the driver's position in the qualifiying session and not the final qualifiying position in for the overall field.

The dataset contains the grid column already whcich records the driver's starting position in the race, which can act as a measure of how a driver qualified (it does however include penalites, which would make it differ from actualy qualifying postions)

In [49]:
# Race-results dataframe
# Keep only Grand Prix race results
race_results = f1_results[
    f1_results["type"] == "R"
].copy()

# Keep the columns relevant to our analysis
race_results = race_results[
    [
        "year",
        "name",
        "date",
        "forename",
        "surname",
        "team_name",
        "grid",
        "position",
        "is_classified",
        "laps_completed",
        "status"
    ]
]

# Create a full driver name
race_results["driver"] = (
    race_results["forename"] + " " + race_results["surname"]
)

# Add the regulation era
race_results["era"] = race_results["year"].apply(
    lambda year: "2019–2021" if year <= 2021 else "2022–2024"
)

race_results.head()

,year,name,date,forename,surname,team_name,grid,position,is_classified,laps_completed,status,driver,era
0,2019,Australian Grand Prix,2019-03-17,Valtteri,Bottas,Mercedes,2.0,1.0,t,58.0,0.0,Valtteri Bottas,2019–2021
4,2019,Australian Grand Prix,2019-03-17,Lewis,Hamilton,Mercedes,1.0,2.0,t,58.0,0.0,Lewis Hamilton,2019–2021
8,2019,Australian Grand Prix,2019-03-17,Max,Verstappen,Red Bull,4.0,3.0,t,58.0,0.0,Max Verstappen,2019–2021
12,2019,Australian Grand Prix,2019-03-17,Sebastian,Vettel,Ferrari,3.0,4.0,t,58.0,0.0,Sebastian Vettel,2019–2021
16,2019,Australian Grand Prix,2019-03-17,Charles,Leclerc,Ferrari,5.0,5.0,t,58.0,0.0,Charles Leclerc,2019–2021


In [50]:
print("Total race entries:", len(race_results))

print("\nMissing starting-grid positions:")
print(race_results["grid"].isna().sum())

print("\nMissing finishing positions:")
print(race_results["position"].isna().sum())

print("\nMissing grid or finishing position:")
print(
    race_results[["grid", "position"]]
    .isna()
    .any(axis=1)
    .sum()
)

print("\nRace entries by year:")
print(race_results.groupby("year").size())

Total race entries: 2559

Missing starting-grid positions:
0

Missing finishing positions:
0

Missing grid or finishing position:
0

Race entries by year:
year
2019    420
2020    340
2021    440
2022    440
2023    440
2024    479
dtype: int64


### Race Data Quality Check

After filtering the combined dataset to include only Grand Prix race sessions (`R`), the dataset contained 2,559 driver-race entries covering 2019–2024.

There were no missing values in the starting-grid (`grid`) or finishing-position (`position`) columns. This means all entries have both values available for the planned comparison.

The number of entries varies by season. In particular, 2024 contains 479 entries, compared with 440 in each of 2021, 2022 and 2023. I will investigate this variation to check for duplicate driver-race records before proceeding with the analysis.

In [51]:
# Check for whether a driver appears more than once in the same race

duplicate_races = race_results.duplicated(
    subset=["year", "name", "driver"],
    keep=False
)

print("Potential duplicate driver-race records:",
      duplicate_races.sum())

race_results[duplicate_races].sort_values(
    ["year", "name", "driver"]
).head(20)

Potential duplicate driver-race records: 0


,year,name,date,forename,surname,team_name,grid,position,is_classified,laps_completed,status,driver,era


In [52]:
print(
    race_results.groupby("year")["name"].nunique()
)

year
2019    21
2020    17
2021    22
2022    22
2023    22
2024    24
Name: name, dtype: int64


In [56]:
race_results[race_results["year"] == 2024].groupby("name").size()
# I noticed that 2024 had one less entrant than expected, but this i was due to a did not start

name
Abu Dhabi Grand Prix         20
Australian Grand Prix        19
Austrian Grand Prix          20
Azerbaijan Grand Prix        20
Bahrain Grand Prix           20
Belgian Grand Prix           20
British Grand Prix           20
Canadian Grand Prix          20
Chinese Grand Prix           20
Dutch Grand Prix             20
Emilia Romagna Grand Prix    20
Hungarian Grand Prix         20
Italian Grand Prix           20
Japanese Grand Prix          20
Las Vegas Grand Prix         20
Mexico City Grand Prix       20
Miami Grand Prix             20
Monaco Grand Prix            20
Qatar Grand Prix             20
Saudi Arabian Grand Prix     20
Singapore Grand Prix         20
Spanish Grand Prix           20
São Paulo Grand Prix         20
United States Grand Prix     20
dtype: int64

### Duplicate Checks

I checked for duplicate driver-race records using the season, Grand Prix name and driver name.

The number of Grands Prix varied between seasons, from 17 in 2020 to 24 in 2024. This explains the variation in the number of driver-race entries across the dataset.


## 4. Analytical questions

The 5 main analytical questions I'm asking are:

    1. How strongly is starting-grid position related to finishing position, and did this relationship change after 2022?
    
    2. How often does the driver starting first go on to win the race?
    
    3. Which drivers gained the most positions during races?

    4. Which constructors performed most consistently across the six seasons?

name
Abu Dhabi Grand Prix         20
Australian Grand Prix        19
Austrian Grand Prix          20
Azerbaijan Grand Prix        20
Bahrain Grand Prix           20
Belgian Grand Prix           20
British Grand Prix           20
Canadian Grand Prix          20
Chinese Grand Prix           20
Dutch Grand Prix             20
Emilia Romagna Grand Prix    20
Hungarian Grand Prix         20
Italian Grand Prix           20
Japanese Grand Prix          20
Las Vegas Grand Prix         20
Mexico City Grand Prix       20
Miami Grand Prix             20
Monaco Grand Prix            20
Qatar Grand Prix             20
Saudi Arabian Grand Prix     20
Singapore Grand Prix         20
Spanish Grand Prix           20
São Paulo Grand Prix         20
United States Grand Prix     20
dtype: int64